# Variation spaces

> Spaces that hold a value, to describe an env's factors of variation (as in stable-worldmodel).

In [ ]:
#| default_exp spaces

In [ ]:
#| hide
from nbdev.showdoc import *

An env's **variation space** is a (nested) `Dict` of these spaces, one per factor of variation
(colors, sizes, numbers of obstacles...). Each space holds a current `value`, reset to its
`init_value` at every env reset, and can be resampled (with an optional `constrain_fn`, by
rejection sampling) or set explicitly.

Envs call `reset_variation_space` in `reset`, which reads the reset options:

* ``options={'variation': ['wall.color', 'obstacles.number']}`` resamples those factors
  (``['all']`` resamples every factor)
* ``options={'variation_values': {'wall.color': 3}}`` sets them

Same API as stable-worldmodel's `spaces` module, from which this code is adapted.

In [ ]:
#| export
# Adapted from stable-worldmodel's `spaces.py` (https://github.com/galilai-group/stable-worldmodel):
#
# MIT License
#
# Copyright (c) 2026 GalilAI-group
#
# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:
#
# The above copyright notice and this permission notice shall be included in all
# copies or substantial portions of the Software.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
# SOFTWARE.

from __future__ import annotations

import logging
import time
from typing import Any, Callable, Generator, Iterable, Sequence

import numpy as np
from gymnasium import spaces

logger = logging.getLogger(__name__)

In [ ]:
#| export
def get_in(mapping: Any, path: Iterable[str]) -> Any:
    "The value at the end of `path` in nested mappings, e.g. ``get_in(space, ['wall', 'color'])``."
    current = mapping
    for key in list(path):
        current = current[key]
    return current


def reset_variation_space(
    variation_space: spaces.Space,
    seed: int | None = None,
    options: dict | None = None,
    default_variations: Iterable[str] | None = None):
    """
    Reset a variation space for an env reset: every factor goes back to its initial value, then
    the factors named in ``options['variation']`` (default: `default_variations`) are resampled
    and those in ``options['variation_values']`` are set.
    """
    variation_space.seed(seed)
    variation_space.reset()
    options = options or {}
    keys = options.get('variation', default_variations or ())
    if not isinstance(keys, Sequence) or isinstance(keys, str):
        raise ValueError("the 'variation' option must be a sequence of variation names")
    variation_space.update(keys)
    if 'variation_values' in options:
        variation_space.set_value(options['variation_values'])
    assert variation_space.check(debug=True), "Variation values must be within the variation space"

In [ ]:
#| export
class _ValueMixin:
    "A space holding a current value (reset to `init_value`), sampled with rejection of `constrain_fn`."

    def _init_value_state(self, init_value, constrain_fn):
        self.constrain_fn = constrain_fn or (lambda x: True)
        self._init_value = init_value
        self._value = init_value

    @property
    def init_value(self) -> Any:
        "The value the space takes at every reset."
        return self._init_value

    @property
    def value(self) -> Any:
        "The current value."
        return self._value

    def reset(self):
        "Set the current value back to the initial value."
        self._value = self._init_value

    def contains(self, x: Any) -> bool:
        return super().contains(x) and bool(self.constrain_fn(x))

    def check(self) -> bool:
        "Whether the current value is valid."
        if not self.constrain_fn(self.value):
            logger.warning(f"{type(self).__name__}: value {self.value} does not satisfy constrain_fn")
            return False
        return super().contains(self.value)

    def sample(self, mask: Any | None = None, max_tries: int = 1000, warn_after_s: float | None = 5.0,
               set_value: bool = True, **kwargs) -> Any:
        "A random value satisfying `constrain_fn` (which becomes the current value if `set_value`)."
        start = time.time()
        for _ in range(max_tries):
            sample = super().sample(mask=mask)
            if self.contains(sample):
                if set_value:
                    self._value = sample
                return sample
            if warn_after_s is not None and time.time() - start > warn_after_s:
                logger.warning("rejection sampling is taking a while...")
        raise RuntimeError(f"rejection sampling: predicate not satisfied after {max_tries} draws")

    def set_init_value(self, value: Any):
        if not self.contains(value):
            raise ValueError(f"Value {value} is not contained in the {type(self).__name__} space")
        self._init_value = value

    def set_value(self, value: Any):
        if not self.contains(value):
            raise ValueError(f"Value {value} is not contained in the {type(self).__name__} space")
        self._value = value


class Discrete(_ValueMixin, spaces.Discrete):
    "Discrete space holding a value."
    def __init__(self, n: int, init_value: int | None = None,
                 constrain_fn: Callable[[int], bool] | None = None, **kwargs):
        spaces.Discrete.__init__(self, n, **kwargs)
        self._init_value_state(init_value, constrain_fn)


class MultiDiscrete(_ValueMixin, spaces.MultiDiscrete):
    "MultiDiscrete space holding a value."
    def __init__(self, nvec: Any, init_value: Any | None = None,
                 constrain_fn: Callable[[Any], bool] | None = None, **kwargs):
        spaces.MultiDiscrete.__init__(self, nvec, **kwargs)
        self._init_value_state(None if init_value is None else np.asarray(init_value, self.dtype), constrain_fn)

    def set_init_value(self, value: Any):
        super().set_init_value(np.asarray(value, self.dtype))

    def set_value(self, value: Any):
        super().set_value(np.asarray(value, self.dtype))


class Box(_ValueMixin, spaces.Box):
    "Box space holding a value."
    def __init__(self, low: Any, high: Any, shape: Iterable[int] | None = None, init_value: Any | None = None,
                 constrain_fn: Callable[[Any], bool] | None = None, **kwargs):
        spaces.Box.__init__(self, low, high, shape, **kwargs)
        self._init_value_state(None if init_value is None else np.asarray(init_value, self.dtype), constrain_fn)

    def check(self) -> bool:
        if not self.constrain_fn(self.value):
            logger.warning(f"Box: value {self.value} does not satisfy constrain_fn")
            return False
        return self.contains(self.value)


class RGBBox(Box):
    "Box space of RGB colors (uint8)."
    def __init__(self, shape: Iterable[int] = (3,), init_value: Any | None = None, **kwargs):
        if not any(dim == 3 for dim in shape):
            raise ValueError("shape must have a channel of size 3")
        super().__init__(low=0, high=255, shape=shape, dtype='uint8', init_value=init_value, **kwargs)

In [ ]:
#| export
class Dict(spaces.Dict):
    """
    Nested space of factors of variation. Sub-spaces are sampled in `sampling_order` (so later
    factors can be constrained by earlier ones), and addressed by dotted names, e.g. 'wall.color'.
    """
    def __init__(
        self,
        spaces_dict: dict[Any, spaces.Space] | None = None,
        init_value: dict | None = None,
        constrain_fn: Callable[[dict], bool] | None = None,
        sampling_order: list[str] | None = None,
        **kwargs):
        super().__init__(spaces_dict, **kwargs)
        self.constrain_fn = constrain_fn or (lambda x: True)
        if init_value is not None:
            self.set_init_value(init_value)
        if sampling_order is None:
            self._sampling_order = list(self.spaces.keys())
        else:
            missing = [k for k in self.spaces if k not in sampling_order]
            if missing:
                logger.warning(f"Dict sampling_order is missing keys {missing}, adding them at the end")
            self._sampling_order = list(sampling_order) + missing
        unknown = set(self._sampling_order) - set(self.spaces)
        if unknown:
            raise ValueError(f"sampling_order contains keys not in spaces: {unknown}")

    @property
    def init_value(self) -> dict:
        return {k: v.init_value if hasattr(v, 'init_value') else v.sample() for k, v in self.spaces.items()}

    @property
    def value(self) -> dict:
        for k, v in self.spaces.items():
            if not hasattr(v, 'value'):
                raise ValueError(f"Space {k} of type {type(v)} does not have a value")
        return {k: v.value for k, v in self.spaces.items()}

    def _get_sampling_order(self, parts: tuple[str, ...] = ()) -> Generator[str, None, None]:
        for key in self._sampling_order:
            path = parts + (str(key),)
            yield '.'.join(path)
            subspace = self.spaces[key]
            if isinstance(subspace, Dict):
                yield from subspace._get_sampling_order(path)
            elif isinstance(subspace, spaces.Dict):
                for subkey in subspace.spaces:
                    yield '.'.join(path + (str(subkey),))

    @property
    def sampling_order(self) -> list[str]:
        "Dotted names of every sub-space, in sampling order."
        return list(self._get_sampling_order())

    def reset(self):
        for v in self.spaces.values():
            if hasattr(v, 'reset'):
                v.reset()

    def contains(self, x: Any) -> bool:
        if not isinstance(x, dict):
            return False
        return all(k in x and self.spaces[k].contains(x[k]) for k in self.spaces) and bool(self.constrain_fn(x))

    def check(self, debug: bool = False) -> bool:
        for k, v in self.spaces.items():
            if hasattr(v, 'check') and not v.check():
                if debug:
                    logger.warning(f"Dict: space {k} failed check()")
                return False
        return True

    def names(self) -> list[str]:
        "Dotted names of the leaf factors, e.g. ['agent.color', 'wall.color']."
        def keys(d: dict, parent: str = '') -> Generator[str, None, None]:
            for k, v in d.items():
                name = f'{parent}.{k}' if parent else str(k)
                if isinstance(v, spaces.Dict):
                    yield from keys(v.spaces, name)
                else:
                    yield name
        return list(keys(self.spaces))

    def sample(self, mask: Any | None = None, max_tries: int = 1000, warn_after_s: float | None = 5.0,
               set_value: bool = True, **kwargs) -> dict:
        start = time.time()
        for _ in range(max_tries):
            sample = {k: self.spaces[k].sample(mask=mask[k] if isinstance(mask, dict) and k in mask else None,
                                               set_value=set_value, **kwargs)
                      for k in self._sampling_order}
            if self.contains(sample):
                return sample
            if warn_after_s is not None and time.time() - start > warn_after_s:
                logger.warning("rejection sampling is taking a while...")
        raise RuntimeError(f"constrain_fn not satisfied after {max_tries} draws")

    def update(self, keys: Iterable[str]):
        "Resample the factors named in `keys` (``['all']``: every factor), in sampling order."
        keys = set(keys)
        if keys == {'all'}:
            self.sample()
        else:
            unknown = keys - set(self.sampling_order)
            if unknown:
                raise ValueError(f"Unknown variations {sorted(unknown)}, available: {self.names()}")
            for name in filter(keys.__contains__, self.sampling_order):
                get_in(self, name.split('.')).sample()
        assert self.check(debug=True), "Values must be within the space"

    def _leaf(self, name: str) -> spaces.Space:
        try:
            return get_in(self, name.split('.'))
        except (KeyError, TypeError):
            raise ValueError(f"Variation {name!r} not found, available: {self.names()}") from None

    def set_init_value(self, variations_values: dict):
        "Set the initial values of factors, ``{'wall.color': 3, ...}``."
        for name, value in variations_values.items():
            self._leaf(name).set_init_value(value)

    def set_value(self, variations_values: dict):
        "Set the current values of factors, ``{'wall.color': 3, ...}``."
        for name, value in variations_values.items():
            self._leaf(name).set_value(value)

    def to_str(self) -> str:
        "The tree of factors, as text."
        def tree(d: dict, indent: int = 0) -> str:
            lines = []
            for k, v in d.items():
                if isinstance(v, spaces.Dict):
                    lines += ['    ' * indent + f'{k}:', tree(v.spaces, indent + 1)]
                else:
                    lines.append('    ' * indent + f'{k}: {v}')
            return '\n'.join(lines)
        return tree(self.spaces)

### Tests

In [ ]:
from fastcore.test import test_fail

space = Dict({
    'wall': Dict({'color': Discrete(6, init_value=5)}),
    'agent': Dict({'color': MultiDiscrete([6, 6], init_value=[0, 1], constrain_fn=lambda c: len(set(c)) == len(c))}),
    'size': Box(0.0, 1.0, shape=(), init_value=0.5),
})
assert space.names() == ['agent.color', 'size', 'wall.color']   # gymnasium's Dict sorts its keys
reset_variation_space(space, seed=0)
assert space['wall']['color'].value == 5 and list(space['agent']['color'].value) == [0, 1]

reset_variation_space(space, seed=0, options={'variation': ['agent.color']})
colors = space['agent']['color'].value
assert len(set(colors)) == 2 and space['wall']['color'].value == 5
reset_variation_space(space, seed=0, options={'variation': ['agent.color']})
assert np.array_equal(space['agent']['color'].value, colors)         # same seed -> same sample

reset_variation_space(space, seed=1, options={'variation': ['all'], 'variation_values': {'wall.color': 2}})
assert space['wall']['color'].value == 2 and space.check()
reset_variation_space(space)                                          # back to the initial values
assert space['wall']['color'].value == 5

test_fail(lambda: reset_variation_space(space, options={'variation': ['wall.colour']}), contains='Unknown')
test_fail(lambda: space.set_value({'agent.color': [1, 1]}), contains='not contained')
space.set_init_value({'wall.color': 0}); reset_variation_space(space)
assert space['wall']['color'].value == 0

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()